In [ ]:
import os

import pandas as pd
import numpy as np
import re

import random
from math import log, e

from datetime import date
from dateutil.relativedelta import relativedelta

data_path = os.path.join("..", "data")
processed_data_path = os.path.join(data_path, 'processed_data/')

## Import Data

In [ ]:
full_ckd_df = pd.read_csv(os.path.join(data_path, "20260313_ckd_data_for_fe.csv"), dtype=object)

In [ ]:
date_cols = ['dateOfBirth', 'dateOfDeath', 'InclusionDate', 'EndpointDate', 'StartDate', 'EndDate']

for col in date_cols:
    full_ckd_df[col] = pd.to_datetime(full_ckd_df[col])

In [ ]:
float_cols = ['Height', 'Weight', 'BMI', 'BMIMeanChange', 'AgeAtDeath', 'ArterialPressure', 'BPDiastolic', 'BPSystolic', 'Albumin',
              'CRP', 'Cholesterol', 'Creatinine', 'Haemoglobin', 'HbA1c', 'NTproBNP', 'RandomGlucose', 'TroponinI', 'UrineACR',
              'UrinePCR', 'eGFR', 'eGFRSlope', 'GFRCalculated', 'GFRCalculatedSlope', 'KFRE', 'ArterialPressureMeanChange',
              'BPDiastolicMeanChange', 'BPSystolicMeanChange', 'AlbuminMeanChange', 'CRPMeanChange', 'CholesterolMeanChange',
              'CreatinineMeanChange', 'HaemoglobinMeanChange', 'HbA1cMeanChange', 'NTproBNPMeanChange', 'RandomGlucoseMeanChange',
              'TroponinIMeanChange', 'UrineACRMeanChange', 'UrinePCRMeanChange', 'eGFRMeanChange', 'GFRCalculatedMeanChange'
             ]

for col in float_cols:
    full_ckd_df[col] = full_ckd_df[col].astype(float)
    full_ckd_df[col] = full_ckd_df[col].apply(lambda x: round(x, 6))

In [ ]:
int_cols = ['AgeAtInclusion', 'AgeAtEndpoint', 'IMDDecile', 'IMDRank', 'grouping', 'GSTTDialysisDistricts', 'GFRSlopeDistricts', 'HypertensiveDisorder',
            'DiabetesMellitus', 'HeartDisease', 'IschemicHeartDisease', 'HeartFailure', 'CerebrovascularDisease', 'CerebrovascularAccident', 'MentalDisorder', 'DepressiveDisorder',
            'Anxiety', 'Schizophrenia', 'Addiction', 'Frailty', 'PeripheralVascularDisease', 'AcuteRenalFailureSyndrome', 'ChronicKidneyDisease', 'PeripheralEdema',
            'PeripheralNerveDisease', 'VisualImpairment', 'ImpairedMobility', 'Dyspnea', 'RetinopathyDueToDiabetesMellitus', 'PulmonaryEdema', 'UlcerOfFootDueToDiabetesMellitus',
            'MyocardialInfarction', 'OrganicMentalDisorder', 'BardetBiedlSyndrome', 'DialysisProcedure', 'AccidentAndEmergencyActivitiesCount', 'IntensiveCareActivitiesCount',
            'OutpatientAttendanceCount', 'OutpatientDNACount', 'OutpatientCancelledCount', 'AKIActivitiesCount', 'ARB', 'ACEInhibitors', 'BetaBlocker', 'CCB', 'Clopidogrel', 'DPP4',
            'Doxazosin', 'Entresto', 'Ezetimibe', 'GLP1', 'Gliclazide', 'Hydralazine', 'Insulin', 'Isosorbide', 'MRA', 'Metformin', 'Minoxidil', 'Moxonidine', 'SLGT2i', 'Statin',
            'priorARB', 'priorAceInhibitors', 'priorBetaBlocker', 'priorCCB', 'priorClopidogrel', 'priorDPP4', 'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1',
            'priorGliclazide', 'priorHydralazine', 'priorInsulin', 'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin']

for col in int_cols:
    full_ckd_df[col] = full_ckd_df[col].apply(lambda x: pd.NA if pd.isnull(x) else int(float(x)))

In [ ]:
str_cols = ['MasterPersonID', 'EndpointCode', 'EndpointCodeType', 'AccessDetails', 'postcode', 'PostcodeDistrict',
            'PostcodeRegion', 'BMICategorisation', 'BPCategorisation', 'eGFRClassification', 'CalculatedGFRClassification'
           ]

for col in str_cols:
    full_ckd_df[col] = full_ckd_df[col].astype(str)

In [ ]:
full_ckd_df['EndpointType'] = full_ckd_df['EndpointType'].apply(lambda x: np.NAN if pd.isnull(x) else str(x))

## Feature Engineering

#### Comorbidities - Keep as they are

- HypertensiveDisorder
- DiabetesMellitus
- HeartDisease
- IschemicHeartDisease
- HeartFailure
- CerebrovascularDisease
- CerebrovascularAccident
- MentalDisorder
- DepressiveDisorder
- Anxiety
- Schizophrenia
- Addiction
- Frailty
- PeripheralVascularDisease
- AcuteRenalFailureSyndrome
- ChronicKidneyDisease
- PeripheralEdema
- PeripheralNerveDisease
- VisualImpairment
- ImpairedMobility
- Dyspnea
- RetinopathyDueToDiabetesMellitus
- PulmonaryEdema
- UlcerOfFootDueToDiabetesMellitus
- MyocardialInfarction
- OrganicMentalDisorder
- BardetBiedlSyndrome

#### Social, Behavioural & Demographic - One-hot encoding

- LivesAlone
- occupation
- SmokingStatus
- HeavyAlcoholUSE
- gender
- race

### Social, Behavioural and Demographic Dummy Column Creation

In [ ]:
dummy_cols = ['LivesAlone', 'occupation', 'SmokingStatus', 'HeavyAlcoholUSE', 'gender', 'race']

full_ckd_df = pd.get_dummies(full_ckd_df, columns=dummy_cols, dtype=int)

del dummy_cols

full_ckd_df.head()

### Aggregation

In [ ]:
agg = {}
rename_dict = {}

sum_cols = ['AccidentAndEmergencyActivitiesCount', 'IntensiveCareActivitiesCount', 'OutpatientAttendanceCount', 'OutpatientDNACount', 'OutpatientCancelledCount',
            'AKIActivitiesCount', 'ARB', 'ACEInhibitors', 'BetaBlocker', 'CCB', 'Clopidogrel', 'DPP4', 'Doxazosin', 'Entresto', 'Ezetimibe', 'GLP1', 'Gliclazide',
            'Hydralazine', 'Insulin', 'Isosorbide', 'MRA', 'Metformin', 'Minoxidil', 'Moxonidine', 'SLGT2i', 'Statin']

for col in sum_cols:
    agg[col] = 'sum'
    rename_dict[col] = f'total{col}'

mean_cols = ['BMI', 'ArterialPressure', 'BPDiastolic', 'BPSystolic', 'Albumin', 'CRP', 'Cholesterol', 'Creatinine', 'Haemoglobin',
             'HbA1c', 'NTproBNP', 'RandomGlucose', 'TroponinI', 'UrineACR', 'UrinePCR', 'GFRCalculated', 'GFRCalculatedSlope', 'KFRE']
 
for col in mean_cols:
    agg[col] = 'mean'
    rename_dict[col] = f'mean{col}'

ckd_agg_df = full_ckd_df.groupby('MasterPersonID').aggregate(agg).reset_index()

### Create Final DataFrame

In [ ]:
join_cols = ['MasterPersonID', 'AgeAtInclusion', 'IMDDecile', 'IMDRank', 'EndpointType', 'HypertensiveDisorder', 'DiabetesMellitus', 'HeartDisease', 'IschemicHeartDisease',
             'HeartFailure', 'CerebrovascularDisease', 'CerebrovascularAccident', 'MentalDisorder', 'DepressiveDisorder', 'Anxiety', 'Schizophrenia', 'Addiction', 'Frailty',
             'PeripheralVascularDisease', 'AcuteRenalFailureSyndrome', 'ChronicKidneyDisease', 'PeripheralEdema', 'PeripheralNerveDisease', 'VisualImpairment', 'ImpairedMobility',
             'Dyspnea', 'RetinopathyDueToDiabetesMellitus', 'PulmonaryEdema', 'UlcerOfFootDueToDiabetesMellitus', 'MyocardialInfarction', 'OrganicMentalDisorder', 'BardetBiedlSyndrome',
             'priorARB', 'priorAceInhibitors', 'priorBetaBlocker', 'priorCCB', 'priorClopidogrel', 'priorDPP4', 'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1',
             'priorGliclazide', 'priorHydralazine', 'priorInsulin', 'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin',
             'BMIMeanChange', 'ArterialPressureMeanChange', 'BPDiastolicMeanChange', 'BPSystolicMeanChange', 'AlbuminMeanChange', 'CRPMeanChange', 'CholesterolMeanChange',
             'CreatinineMeanChange', 'HaemoglobinMeanChange', 'HbA1cMeanChange', 'NTproBNPMeanChange', 'RandomGlucoseMeanChange', 'TroponinIMeanChange', 'UrineACRMeanChange',
             'UrinePCRMeanChange', 'GFRCalculatedMeanChange', 'LivesAlone_No', 'LivesAlone_Yes', 'occupation_Employed', 'occupation_Other', 'occupation_Retired',
             'occupation_Unemployed', 'SmokingStatus_Current Smoker', 'SmokingStatus_Ex-Smoker', 'SmokingStatus_Non-Smoker', 'SmokingStatus_Occasional Smoker', 'HeavyAlcoholUSE_No',
             'HeavyAlcoholUSE_Yes', 'gender_Female', 'gender_Male', 'race_Asian', 'race_Black', 'race_Not Stated', 'race_Other', 'race_White']

ckd_df = full_ckd_df[join_cols].merge(ckd_agg_df, how='left', on='MasterPersonID')

for col in mean_cols:
    ckd_df[col] = ckd_df[col].apply(lambda x: round(x, 2))
    del col

ckd_df = ckd_df.drop_duplicates().reset_index(drop=True).rename(columns=rename_dict)

del ckd_agg_df, join_cols, sum_cols, mean_cols, agg, rename_dict

### Include First & Last GFR Classifications

In [ ]:
gfr_classifications_df = full_ckd_df[full_ckd_df['CalculatedGFRClassification']!='nan'][['MasterPersonID', 'CalculatedGFRClassification']]

first_gfr_classification_df = gfr_classifications_df.groupby('MasterPersonID')['CalculatedGFRClassification'].first().reset_index()
first_gfr_classification_df.columns = ['MasterPersonID', 'firstGFRClassification']

last_gfr_classification_df = gfr_classifications_df.groupby('MasterPersonID')['CalculatedGFRClassification'].last().reset_index()
last_gfr_classification_df.columns = ['MasterPersonID', 'lastGFRClassification']

ckd_df = ckd_df.merge(first_gfr_classification_df, how='left', on='MasterPersonID').merge(last_gfr_classification_df, how='left', on='MasterPersonID')

del gfr_classifications_df, first_gfr_classification_df, last_gfr_classification_df

### Include First & Last BMI Classifications

In [ ]:
bmi_classifications_df = full_ckd_df[full_ckd_df['BMICategorisation']!='nan'][['MasterPersonID', 'BMICategorisation']]

first_bmi_classification_df = bmi_classifications_df.groupby('MasterPersonID')['BMICategorisation'].first().reset_index()
first_bmi_classification_df.columns = ['MasterPersonID', 'firstBMICategorisation']

last_bmi_classification_df = bmi_classifications_df.groupby('MasterPersonID')['BMICategorisation'].last().reset_index()
last_bmi_classification_df.columns = ['MasterPersonID', 'lastBMICategorisation']

ckd_df = ckd_df.merge(first_bmi_classification_df, how='left', on='MasterPersonID').merge(last_bmi_classification_df, how='left', on='MasterPersonID')

del bmi_classifications_df, first_bmi_classification_df, last_bmi_classification_df

### Include First & Last Blood Pressure Classifications

In [ ]:
bp_classifications_df = full_ckd_df[full_ckd_df['BPCategorisation']!='nan'][['MasterPersonID', 'BPCategorisation']]

first_bp_classification_df = bp_classifications_df.groupby('MasterPersonID')['BPCategorisation'].first().reset_index()
first_bp_classification_df.columns = ['MasterPersonID', 'firstBPCategorisation']

last_bp_classification_df = bp_classifications_df.groupby('MasterPersonID')['BPCategorisation'].last().reset_index()
last_bp_classification_df.columns = ['MasterPersonID', 'lastBPCategorisation']

ckd_df = ckd_df.merge(first_bp_classification_df, how='left', on='MasterPersonID').merge(last_bp_classification_df, how='left', on='MasterPersonID')

del bp_classifications_df, first_bp_classification_df, last_bp_classification_df

### Finalise CKD DataFrame for Modelling

In [ ]:
def kfre(row):
    if not pd.isnull(row['meanUrineACR']) and not pd.isnull(row['meanGFRCalculated']) and row['meanUrineACR']!=0:
        age = row['AgeAtInclusion']
        gfr = row['meanGFRCalculated']
        acr = row['meanUrineACR']

        if row['gender_Female'] == 1:
            sex = 0
        else:
            sex = 1

        return 1-0.957**e**((-0.2201*(age/10-7.036))+(0.2467*(sex-0.5642))-(0.5567*(gfr/5-7.222))+(0.451*(log(acr/0.113)-5.137)))

In [ ]:
cols = ['MasterPersonID', 'AgeAtInclusion', 'LivesAlone_No', 'LivesAlone_Yes', 'occupation_Employed', 'occupation_Other', 'occupation_Retired', 'occupation_Unemployed',
        'SmokingStatus_Current Smoker', 'SmokingStatus_Ex-Smoker', 'SmokingStatus_Non-Smoker', 'SmokingStatus_Occasional Smoker', 'HeavyAlcoholUSE_No', 'HeavyAlcoholUSE_Yes',
        'gender_Female', 'gender_Male', 'race_Asian', 'race_Black', 'race_Not Stated', 'race_Other', 'race_White', 'IMDDecile', 'IMDRank', 'HypertensiveDisorder',
        'DiabetesMellitus', 'HeartDisease', 'IschemicHeartDisease', 'HeartFailure', 'CerebrovascularDisease', 'CerebrovascularAccident', 'MentalDisorder', 'DepressiveDisorder',
        'Anxiety', 'Schizophrenia', 'Addiction', 'Frailty', 'PeripheralVascularDisease', 'AcuteRenalFailureSyndrome', 'ChronicKidneyDisease', 'PeripheralEdema',
        'PeripheralNerveDisease', 'VisualImpairment', 'ImpairedMobility', 'Dyspnea', 'PulmonaryEdema', 'MyocardialInfarction', 'OrganicMentalDisorder', 'BardetBiedlSyndrome',
        'priorARB', 'priorAceInhibitors', 'priorBetaBlocker', 'priorCCB', 'priorClopidogrel', 'priorDPP4', 'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1',
        'priorGliclazide', 'priorHydralazine', 'priorInsulin', 'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin',
        'BMIMeanChange', 'ArterialPressureMeanChange', 'BPDiastolicMeanChange', 'BPSystolicMeanChange', 'AlbuminMeanChange', 'CRPMeanChange', 'CholesterolMeanChange',
        'CreatinineMeanChange', 'HaemoglobinMeanChange', 'HbA1cMeanChange', 'NTproBNPMeanChange', 'RandomGlucoseMeanChange', 'TroponinIMeanChange', 'UrineACRMeanChange',
        'UrinePCRMeanChange', 'GFRCalculatedMeanChange','totalAccidentAndEmergencyActivitiesCount', 'totalIntensiveCareActivitiesCount', 'totalOutpatientAttendanceCount',
        'totalOutpatientDNACount', 'totalOutpatientCancelledCount', 'totalAKIActivitiesCount', 'totalARB', 'totalACEInhibitors', 'totalBetaBlocker', 'totalCCB', 'totalClopidogrel',
        'totalDPP4', 'totalDoxazosin', 'totalEntresto', 'totalEzetimibe', 'totalGLP1', 'totalGliclazide', 'totalHydralazine', 'totalInsulin', 'totalIsosorbide', 'totalMRA',
        'totalMetformin', 'totalMinoxidil', 'totalMoxonidine', 'totalSLGT2i', 'totalStatin', 'meanBMI', 'meanArterialPressure', 'meanBPDiastolic', 'meanBPSystolic', 'meanAlbumin',
        'meanCRP', 'meanCholesterol', 'meanCreatinine', 'meanHaemoglobin', 'meanHbA1c', 'meanNTproBNP', 'meanRandomGlucose', 'meanTroponinI', 'meanUrineACR', 'meanUrinePCR',
        'meanGFRCalculated', 'meanGFRCalculatedSlope', 'meanKFRE', 'firstGFRClassification', 'lastGFRClassification', 'firstBMICategorisation', 'lastBMICategorisation',
        'firstBPCategorisation', 'lastBPCategorisation', 'EndpointType', 'Endpoint']

In [ ]:
rename_dict = {'SmokingStatus_Current Smoker' : 'SmokingStatus_CurrentSmoker',
               'SmokingStatus_Ex-Smoker' : 'SmokingStatus_ExSmoker',
               'SmokingStatus_Non-Smoker' : 'SmokingStatus_NonSmoker',
               'SmokingStatus_Occasional Smoker' : 'SmokingStatus_OccasionalSmoker'}

In [ ]:
ckd_df['Endpoint'] = ckd_df['EndpointType'].apply(lambda x: 0 if pd.isnull(x) else 1)
ckd_df['meanKFRE'] = ckd_df['meanKFRE'].combine_first(ckd_df.apply(lambda row: kfre(row), axis=1))

ckd_df = ckd_df[cols].rename(columns=rename_dict)

ckd_df.head()

## Export Final Dataset

In [ ]:
# --- Save Results ---
file_name = "20260313_ckd_data_for_ml.csv"

ckd_df.to_csv(os.path.join(data_path, file_name), index=False)
print("✅ Results saved.")